# Rule: **build_industrial_production_per_node**


**Description**

This rule builds the industrial production per model region (node). This is done by multiplying the previously built sectoral industrial distribution keys (with values of 0-1) and the future industrial production at national level:

**future production(node, sector) = future production(country, sector) × distribution key(sector, node)**

The sum of nodal production must equal country-level production. The output of this rule is a spatially explicit industrial production datase.

**Inputs**

- resources/{prefix}/{name}/`industrial_distribution_key_base_s_{clusters}.csv`
- resources/{prefix}/{name}/`industrial_production_per_country_tomorrow_{horizon}.csv`

**Outputs**

- resources/{prefix}/{name}/`industrial_production_base_s_{clusters}_{horizon}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
clusters = '' # number of clusters or 'adm'
horizon = ''

### Spatial domain 'ES' or 'EU' (for maps domain and NUTS regions)
spatial_domain = ''

In [ ]:
##### Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os 
import sys
from matplotlib.lines import Line2D
import cartopy.crs as ccrs

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Region files
region_tag = f"base_s_{clusters}"
gdf_regions_onshore, gdf_regions_offshore = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=region_tag,
)

##### Set options
pd.set_option("display.max_columns", None)

## `industrial_production_base_s_{clusters}_{horizon}.csv`  
Load the file and preview its content.

In [ ]:
file = f"industrial_production_base_s_{clusters}_{horizon}.csv"

ind_prod_node_tomorrow = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

ind_prod_node_tomorrow.head()

What is the spatial distribution for the horizon year?

In [ ]:
#################### Prepare data

df = ind_prod_node_tomorrow.copy()
df.columns = df.columns.str.strip()

node_col = df.columns[0]

sector_cols = df.columns[1:]

df[sector_cols] = df[sector_cols].apply(pd.to_numeric, errors="coerce")

df["total_production"] = df[sector_cols].sum(axis=1)

# Merge regions
gdf = gdf_regions_onshore.merge(
    df,
    left_on="name",
    right_on=node_col,
    how="left",
)

gdf[sector_cols] = gdf[sector_cols].fillna(0)

#################### Plot


fig, ax = plt.subplots(
    figsize=(15, 10),
    subplot_kw={"projection": ccrs.PlateCarree()}
)


### Add map features
xp.map_add_features(ax, params['map_add_features'])

gdf.plot(
    ax=ax,
    color="white",
    edgecolor="black",
    linewidth=0.5,
)

max_prod = gdf["total_production"].max()

# Adjust the radius scale to modify the overall size of the piecharts
radius_scale = 0.7

colors = plt.cm.tab20(np.linspace(0, 1, len(sector_cols)))

for _, row in gdf.iterrows():
    if pd.isna(row["total_production"]) or row["total_production"] <= 0:
        continue

    sizes = row[sector_cols].values.astype(float)
    sizes = np.where(sizes < 0, 0, sizes)

    if sizes.sum() == 0:
        continue

    x = row.geometry.centroid.x
    y = row.geometry.centroid.y

    r = radius_scale * np.sqrt(row["total_production"] / max_prod)

    ax.pie(
        sizes,
        radius=r,
        center=(x, y),
        colors=colors,
        wedgeprops=dict(linewidth=0),
    )



legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        label=sector,
        markerfacecolor=colors[i],
        markersize=8
    )
    for i, sector in enumerate(sector_cols)
]

# Legend settings
ax.legend(
    handles=legend_elements,
    title="Sector",
    fontsize=10,
    title_fontsize=12,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

bounds = params[f'boundaries_onshore_{spatial_domain}']

ax.set_xlim(bounds[0], bounds[1])
ax.set_ylim(bounds[2], bounds[3])

ax.axis("off")

# Title settings
ax.set_title(
    f"Projected industrial production ({spatial_domain}, {horizon})",
    fontsize=15
)